# Trained heads results

Phase recognition with heads trained on frozen encoder embeddings (HecVL, SurgVLP, PeskaVLP, SurgCLIP, SurgMotion300M): per-frame heads (linear, simple MLP) and the causal temporal convolutional sequence head, single seeds and seed ensembles. Reads evaluation CSVs and per-frame prediction files only; heads are trained and evaluated leave-one-surgery-out through the `evaluate` task. Every section shows the ground truth (`none`, the seven phases of `assets/phases/default.json`) and each merge variant in `MERGE_SCHEMES`, and compares against the best zero-shot results of notebook 3.

## Setup

Imports, paths, models, head variants, schemes and sampling rates.

In [1]:
import json
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import entropy
from sklearn.metrics import precision_recall_fscore_support

from surgphase.config import MERGE_SCHEMES, MODELS, PHASE_GROUP_FILE, RESULTS_DIR, ROOT_DIR, UNLABELLED
from surgphase.data import PhaseMapper, load_phase_group, load_surgeries, merge_phases
from surgphase.eval import load_predictions, score
from surgphase.temporal import HMM_METHODS, Decoding, leave_one_out_hmms
from surgphase.viz import CATEGORICAL, NEUTRAL, SEQUENTIAL, phase_colors, plot_confusion_matrix, plot_timeline, style_axes
from surgphase.viz.style import GRIDLINE, INK, MUTED_INK, SECONDARY_INK, SURFACE

HEADS_DIR = RESULTS_DIR / "evaluation" / "heads"
ZEROSHOT_DIR = RESULTS_DIR / "evaluation" / "zeroshot"
FPS = 1.0
ENSEMBLE = "ensemble"
NEUTRAL_PHASE = "P0"
SUMMARY_METRICS = ["accuracy", "f1_macro", "precision", "iou", "edit_score", "f1_at_25"]
COMPARE_METRICS = ["f1_macro", "precision", "iou", "accuracy"]
CONSISTENCY_METRICS = ["transitions", "backward transitions", "edit_score", "f1_at_25"]
CHANCE_SEED = 0
TOP_CONFIGURATIONS = 15

MODEL_COLORS = {model: CATEGORICAL[index] for index, model in enumerate(MODELS)}
BASE_GROUP = load_phase_group(PHASE_GROUP_FILE)
BASE_ORDER = [phase.id for phase in PhaseMapper(BASE_GROUP).phases]
BASE_COLORS = dict(zip(BASE_ORDER, phase_colors(len(BASE_ORDER), neutral_phase=BASE_ORDER.index(NEUTRAL_PHASE)), strict=True))


def evaluate_command(scheme):
    merges = " ".join("+".join(ids) for ids in MERGE_SCHEMES[scheme])
    merge_flag = f"--merge {merges} " if merges else ""
    return (
        "uv run poe evaluate --annotations assets/annotations/default/annotated_frames.csv --fps 1 "
        f"{merge_flag}--models HecVL SurgVLP PeskaVLP SurgCLIP SurgMotion300M --sources frame_head sequence_head "
        "--head-types linear simple --class-weights --seeds 1 2 3 --sequence-stages 3 --sequence-layers 8 "
        "--temporal-methods none moving_average moving_median exponential_moving_average hmm_filter "
        f"--preprocessor letterbox --output-dir results/evaluation/heads/{scheme}"
    )


def phase_label(phase):
    return "+".join(phase.members)


def phase_title(phase):
    return f"{phase_label(phase)} " + " / ".join(BASE_GROUP.phases[member].name for member in phase.members)


def resolve(path):
    path = Path(path)
    return path if path.is_absolute() else ROOT_DIR / path


def decoding_of(row):
    window = None if pd.isna(row.window_seconds) else float(row.window_seconds)
    return Decoding(row.method, window, bool(row.causal))


def decoding_label(decoding):
    if decoding.window_seconds is None:
        return decoding.method
    return f"{decoding.method} {decoding.window_seconds:g}s{'' if decoding.causal else ' centred'}"


def held_out_selection(frame, columns, by=("model",)):
    frame = frame.assign(config=frame[columns].astype(str).agg(" | ".join, axis=1))
    chosen = []
    for _, group in frame.groupby(list(by)) if by else [(None, frame)]:
        scores = group.pivot_table(index="config", columns="surgery_id", values="f1_macro")
        for surgery_id in scores.columns:
            choice = scores.drop(columns=surgery_id).mean(axis=1).idxmax()
            chosen.append(group[(group["surgery_id"] == surgery_id) & (group["config"] == choice)])
    return pd.concat(chosen)


def clean_axes(ax, grid_axis="y"):
    style_axes(ax)
    if grid_axis:
        ax.grid(axis=grid_axis, color=GRIDLINE, linewidth=0.6)
    ax.set_axisbelow(True)


def figure_legend(fig, axes):
    handles = {}
    for ax in np.ravel(axes):
        for handle, label in zip(*ax.get_legend_handles_labels(), strict=True):
            handles.setdefault(label, handle)
    fig.legend(
        list(handles.values()), list(handles), loc="outside lower center", ncol=min(len(handles), 6),
        frameon=False, labelcolor=SECONDARY_INK,
    )


def annotate_heatmap(ax, values, fmt="{:.2f}"):
    for (row, column), value in np.ndenumerate(values):
        if np.isfinite(value):
            ax.text(column, row, fmt.format(value), ha="center", va="center", fontsize=8, color=SURFACE if value > 0.6 else INK)


def heatmap(ax, matrix, xlabels, ylabels, title):
    style_axes(ax)
    ax.imshow(matrix, cmap=SEQUENTIAL, vmin=0, vmax=1, aspect="auto")
    annotate_heatmap(ax, matrix)
    ax.set_xticks(range(len(xlabels)), labels=xlabels, rotation=30, ha="right")
    ax.set_yticks(range(len(ylabels)), labels=ylabels)
    ax.tick_params(length=0)
    ax.set_title(title, loc="left", fontsize=10)


def with_scheme(frames):
    return pd.concat(frames, names=["scheme"])


def transitions(sequence):
    sequence = sequence[sequence != UNLABELLED]
    steps = np.diff(sequence)
    return int(np.count_nonzero(steps)), int(np.count_nonzero(steps < 0))

## Load results

Head rows from the evaluation CSVs, per-frame probabilities from the prediction files and the aligned per-frame phase labels, per scheme. Zero-shot rows from `results/evaluation/zeroshot/<scheme>` for comparison.

In [ ]:
def load_scheme(name):
    run_dir = HEADS_DIR / name
    if not (run_dir / "metrics.csv").is_file():
        print(f"{name}: no run yet, create it with:\n{evaluate_command(name)}")
        return None
    run = json.loads((run_dir / "run.json").read_text(encoding="utf-8"))
    merges = [tuple(ids) for ids in run["merge"]]
    group = merge_phases(load_phase_group(PHASE_GROUP_FILE), merges)
    phases = PhaseMapper(group).phases
    results = pd.read_csv(run_dir / "metrics.csv")
    results = results[results["source"].isin(["frame_head", "sequence_head"]) & (results["fps"] == FPS)].reset_index(drop=True)
    if "seed" not in results:
        results["seed"] = np.nan
    results["seed"] = results["seed"].fillna(ENSEMBLE if results["seed"].isna().all() else "").astype(str)
    results = results.rename(columns={"variant": "head"})
    results["decoding"] = [decoding_label(decoding_of(row)) for row in results.itertuples()]
    heads = list(dict.fromkeys(results.sort_values("source", kind="stable")["head"]))
    surgeries = load_surgeries(resolve(run["annotations"]), group, ROOT_DIR, FPS, run["surgeries"], merges)
    labels = {surgery_id: surgery.labels for surgery_id, surgery in surgeries.items()}
    probabilities = {
        (row.model, row.head, row.seed): load_predictions(run_dir / "predictions" / row.predictions_file)
        for row in results.drop_duplicates(["model", "head", "seed"]).itertuples()
    }
    misaligned = [
        (key, surgery_id) for key, by_id in probabilities.items() for surgery_id, p in by_id.items() if len(p) != len(labels[surgery_id])
    ]
    if misaligned:
        raise ValueError(f"{name}: prediction and label lengths differ for {misaligned}")
    main = results[results["seed"] == ENSEMBLE]
    raw = main[main["method"] == "none"]
    best_head = raw.groupby(["model", "head"])["f1_macro"].mean().groupby("model").idxmax().map(lambda index: index[1])

    def best_rows(frame):
        return frame[[best_head.get(m) == h for m, h in zip(frame["model"], frame["head"], strict=True)]]

    return SimpleNamespace(
        name=name,
        run=run,
        phases=phases,
        phase_ids=[phase_label(phase) for phase in phases],
        phase_names=[phase_title(phase) for phase in phases],
        colors=[BASE_COLORS[phase.members[0]] for phase in phases],
        num_phases=len(phases),
        results=results,
        main=main,
        raw=raw,
        heads=heads,
        seeds=[seed for seed in dict.fromkeys(results["seed"]) if seed != ENSEMBLE],
        surgeries=surgeries,
        labels=labels,
        probabilities=probabilities,
        decoding_order=list(dict.fromkeys(results["decoding"])),
        decodings_by_label={label: decoding_of(row) for label, row in zip(results["decoding"], results.itertuples())},
        best_head=best_head,
        best_keys=[(m, h, ENSEMBLE) for m, h in best_head.items()],
        best_raw=best_rows(raw),
        best_main=best_rows(main),
    )


def pooled(scheme, key):
    by_id = scheme.probabilities[key]
    y = np.concatenate([scheme.labels[surgery_id] for surgery_id in by_id])
    p = np.concatenate([by_id[surgery_id] for surgery_id in by_id])
    keep = y != UNLABELLED
    return y[keep], p[keep]


def load_zeroshot(name):
    path = ZEROSHOT_DIR / name / "metrics.csv"
    if not path.is_file():
        return None
    frame = pd.read_csv(path)
    frame = frame[(frame["source"] == "zeroshot") & (frame["fps"] == FPS)].reset_index(drop=True)
    frame["decoding"] = [decoding_label(decoding_of(row)) for row in frame.itertuples()]
    return frame


schemes = {name: scheme for name in MERGE_SCHEMES if (scheme := load_scheme(name)) is not None}
if not schemes:
    raise FileNotFoundError(f"no head runs under {HEADS_DIR}, create them with the commands above")
zeroshot = {name: frame for name in schemes if (frame := load_zeroshot(name)) is not None}
scheme_names = list(schemes)
models = [model for model in MODELS if any(model in set(scheme.results["model"]) for scheme in schemes.values())]
heads = list(dict.fromkeys(head for scheme in schemes.values() for head in scheme.heads))
HEAD_COLORS = dict(zip(heads, SEQUENTIAL(np.linspace(0.35, 0.95, len(heads))), strict=True))
for name, scheme in schemes.items():
    print(
        f"{name}: {scheme.num_phases} classes ({', '.join(scheme.phase_ids)}), {len(scheme.results):,} rows, "
        f"heads {', '.join(scheme.heads)}, seeds {', '.join(scheme.seeds) or 'single'} + {ENSEMBLE}, "
        f"{len(scheme.decoding_order)} decodings, {scheme.results['surgery_id'].nunique()} surgeries"
        f"{'' if name in zeroshot else ', no zero-shot run'}"
    )
print(f"models: {', '.join(models)}")
print(f"analysis uses the {ENSEMBLE} of the seeds unless a section compares seeds")

## Overall performance

Accuracy, macro F1, precision and IoU per model and head variant without temporal decoding, against chance and majority-class baselines.

In [ ]:
baseline_rows = []
for name, scheme in schemes.items():
    rng = np.random.default_rng(CHANCE_SEED)
    for surgery_id, y in scheme.labels.items():
        others = np.concatenate([other[other != UNLABELLED] for s, other in scheme.labels.items() if s != surgery_id])
        majority = np.full_like(y, np.bincount(others, minlength=scheme.num_phases).argmax())
        uniform = rng.integers(0, scheme.num_phases, size=len(y))
        for baseline, predicted in (("majority", majority), ("uniform", uniform)):
            result = score(y, predicted)
            baseline_rows.append(
                {"scheme": name, "baseline": baseline, "surgery_id": surgery_id, **{m: getattr(result, m) for m in SUMMARY_METRICS}}
            )
baselines = pd.DataFrame(baseline_rows)
baseline_summary = baselines.groupby(["scheme", "baseline"])[SUMMARY_METRICS].mean()

fig, axes = plt.subplots(
    len(schemes), 2, figsize=(12, (0.55 * len(models) + 1.1) * len(schemes) + 0.8), layout="constrained", sharex=True, squeeze=False
)
for row, (name, scheme) in enumerate(schemes.items()):
    plot_models = [m for m in models if m in set(scheme.best_raw["model"])]
    for ax, metric in zip(axes[row], ("accuracy", "f1_macro"), strict=True):
        clean_axes(ax, grid_axis="x")
        for position, model in enumerate(plot_models):
            values = scheme.best_raw.loc[scheme.best_raw["model"] == model, metric].to_numpy()
            ax.scatter(values, np.full(len(values), position), s=28, color=MODEL_COLORS[model], alpha=0.6, linewidth=0)
            ax.vlines(values.mean(), position - 0.3, position + 0.3, color=INK, linewidth=2)
        for baseline, style in (("majority", "--"), ("uniform", ":")):
            ax.axvline(
                baseline_summary.loc[(name, baseline), metric], color=MUTED_INK, linestyle=style, linewidth=1, label=f"{baseline} baseline"
            )
        ax.set_xlim(0, 1)
        ax.set_yticks(range(len(plot_models)), labels=[f"{m}\n{scheme.best_head[m]}" for m in plot_models] if metric == "accuracy" else [])
        ax.set_ylim(len(plot_models) - 0.5, -0.5)
        ax.spines["left"].set_visible(False)
        ax.tick_params(axis="y", length=0, labelcolor=INK)
        ax.set_title(f"{metric}, {name} ({scheme.num_phases} classes)", loc="left", fontsize=10)
for ax in axes[-1]:
    ax.set_xlabel("dot = surgery, line = mean")
figure_legend(fig, axes)
fig.suptitle(f"Trained heads without decoding, best head per model, {ENSEMBLE} @ {FPS:g} fps", x=0.01, ha="left", color=INK)
plt.show()

overall_tables = {}
for name, scheme in schemes.items():
    overall = scheme.best_raw.groupby("model").agg(head=("head", "first"), **{m: (m, "mean") for m in SUMMARY_METRICS})
    overall["f1_macro held-out"] = held_out_selection(scheme.raw, ["head"]).groupby("model")["f1_macro"].mean()
    overall = pd.concat(
        [overall.reindex(models), baseline_summary.loc[name].rename(index=lambda b: f"{b} baseline").rename_axis("model")]
    )
    overall_tables[name] = overall
print(
    "head and scores: head with the highest mean over all surgeries (optimistic); "
    "f1_macro held-out: head chosen per test surgery on the other surgeries"
)
with_scheme(overall_tables).round(3)

## Head variants

Linear, simple MLP and causal TCN heads per model. How much temporal context inside the head adds over per-frame classification.

In [ ]:
head_metrics = ["f1_macro", "edit_score"]
head_tables = {}
fig, axes = plt.subplots(
    len(schemes), len(head_metrics), figsize=(12, 3.2 * len(schemes) + 0.8), layout="constrained", sharey=True, squeeze=False
)
for row, (name, scheme) in enumerate(schemes.items()):
    summary = scheme.raw.groupby(["model", "head"])[head_metrics].agg(["mean", "std"])
    positions = np.arange(len(models))
    width = 0.8 / len(heads)
    for ax, metric in zip(axes[row], head_metrics, strict=True):
        clean_axes(ax)
        for index, head in enumerate(heads):
            values = summary[metric].reindex(pd.MultiIndex.from_product([models, [head]]))
            ax.bar(
                positions - 0.4 + width * (index + 0.5), values["mean"], width, yerr=values["std"], color=HEAD_COLORS[head],
                error_kw={"ecolor": MUTED_INK, "linewidth": 0.8}, label=head,
            )
        ax.set_xticks(positions, labels=models)
        ax.set_ylim(0, 1)
        ax.set_title(f"{metric}, {name}", loc="left", fontsize=10)
    head_tables[name] = summary.xs("mean", axis=1, level=1).unstack("head").reindex(models)
figure_legend(fig, axes)
fig.suptitle(f"Head variants without decoding, {ENSEMBLE}, bar = mean over surgeries, line = std", x=0.01, ha="left", color=INK)
plt.show()

with_scheme(head_tables).round(3)

## Seed ensembles

Single seeds against the averaged probabilities of several seeds: mean, spread across seeds and gain of the ensemble.

In [ ]:
seed_tables = {}
seeded = {name: scheme for name, scheme in schemes.items() if scheme.seeds}
if not seeded:
    print(f"no run trained several seeds, rerun with --seeds:\n{evaluate_command(scheme_names[0])}")
else:
    fig, axes = plt.subplots(
        len(seeded), 1, figsize=(12, 3.4 * len(seeded) + 0.6), layout="constrained", sharey=True, squeeze=False
    )
    for ax, (name, scheme) in zip(axes[:, 0], seeded.items(), strict=True):
        clean_axes(ax)
        raw = scheme.results[scheme.results["method"] == "none"]
        per_seed = raw.groupby(["model", "head", "seed"])["f1_macro"].mean().unstack("seed")
        singles = per_seed[scheme.seeds]
        table = pd.DataFrame(
            {
                "seed mean": singles.mean(axis=1),
                "seed std": singles.std(axis=1),
                "seed best": singles.max(axis=1),
                ENSEMBLE: per_seed[ENSEMBLE],
            }
        )
        table["ensemble gain"] = table[ENSEMBLE] - table["seed mean"]
        seed_tables[name] = table.reindex(models, level="model")
        labels = [f"{m}\n{h}" for m, h in seed_tables[name].index]
        positions = np.arange(len(labels))
        for position, (key, values) in zip(positions, singles.reindex(seed_tables[name].index).iterrows(), strict=True):
            ax.scatter(
                np.full(len(values), position), values, s=22, color=MODEL_COLORS[key[0]], alpha=0.6, linewidth=0,
                label="single seed" if position == 0 else None,
            )
        ax.scatter(positions, seed_tables[name][ENSEMBLE], marker="_", s=260, color=INK, linewidth=2, zorder=3, label=ENSEMBLE)
        ax.set_xticks(positions, labels=labels, fontsize=7)
        ax.set_ylabel("macro F1")
        ax.set_title(f"{name}: seeds {', '.join(scheme.seeds)}", loc="left", fontsize=10)
    figure_legend(fig, axes)
    fig.suptitle("Single seeds against their probability average, no decoding", x=0.01, ha="left", color=INK)
    plt.show()
    print(
        "ensemble gain: ensemble macro F1 minus the mean of the single seeds; "
        f"mean gain over models and heads: {pd.concat(seed_tables.values())['ensemble gain'].mean():.3f}"
    )
with_scheme(seed_tables).round(3) if seed_tables else None

## Per-phase performance

Precision, recall and F1 per phase, model and head. Which phases the heads learn and which they miss.

In [ ]:
phase_rows = []
for name, scheme in schemes.items():
    for key in scheme.best_keys:
        y, p = pooled(scheme, key)
        precision, recall, f1, support = precision_recall_fscore_support(
            y, p.argmax(axis=1), labels=range(scheme.num_phases), zero_division=0
        )
        for index, phase in enumerate(scheme.phase_ids):
            phase_rows.append(
                {
                    "scheme": name, "model": key[0], "head": key[1], "phase": phase, "order": index,
                    "precision": precision[index], "recall": recall[index], "f1": f1[index], "support": support[index],
                }
            )
per_phase = pd.DataFrame(phase_rows)

phase_metrics = ["precision", "recall", "f1"]
for name, scheme in schemes.items():
    fig, axes = plt.subplots(
        1, len(phase_metrics), figsize=(4.2 * len(phase_metrics), 0.45 * scheme.num_phases + 1.8), layout="constrained", squeeze=False
    )
    plot_table = per_phase[per_phase["scheme"] == name]
    for ax, metric in zip(axes[0], phase_metrics, strict=True):
        matrix = plot_table.pivot(index="phase", columns="model", values=metric).reindex(index=scheme.phase_ids)
        matrix = matrix[[m for m in models if m in matrix.columns]]
        heatmap(
            ax, matrix.to_numpy(), matrix.columns, scheme.phase_names if ax is axes[0][0] else [""] * scheme.num_phases, metric
        )
    fig.suptitle(f"Per-phase performance, {name}, best head per model, {ENSEMBLE}, no decoding", x=0.01, ha="left", color=INK)
    plt.show()

(
    per_phase.pivot_table(index=["scheme", "order", "phase"], columns="model", values="f1")
    .reindex(scheme_names, level="scheme")
    .reindex(columns=models)
    .droplevel("order")
    .round(3)
)

## Confusion matrices

Row-normalised confusion matrices per model for the best head.

In [ ]:
for name, scheme in schemes.items():
    fig, axes = plt.subplots(1, len(models), figsize=(3.6 * len(models), 4), layout="constrained", squeeze=False)
    for column, model in enumerate(models):
        ax = axes[0][column]
        key = (model, scheme.best_head.get(model), ENSEMBLE)
        if key not in scheme.probabilities:
            ax.set_visible(False)
            continue
        y, p = pooled(scheme, key)
        plot_confusion_matrix(y, p.argmax(axis=1), scheme.phase_ids, ax=ax)
        ax.set_title(f"{model}\n{key[1]}", loc="left", fontsize=10)
        if column:
            ax.set_ylabel("")
    fig.suptitle(f"Row-normalised confusion matrices, {name}, {ENSEMBLE}, no decoding", x=0.01, ha="left", color=INK)
    plt.show()

## Prediction bias

Predicted phase distribution against the ground truth. Shows whether predictions collapse onto frequent phases.

In [ ]:
bias_tables = {}
for name, scheme in schemes.items():
    counts = np.bincount(np.concatenate([y[y != UNLABELLED] for y in scheme.labels.values()]), minlength=scheme.num_phases)
    true_share = counts / counts.sum()
    rows, predicted_share = [], {}
    for key in scheme.best_keys:
        y, p = pooled(scheme, key)
        share = np.bincount(p.argmax(axis=1), minlength=scheme.num_phases) / len(y)
        predicted_share[key] = share
        rows.append(
            {
                "model": key[0],
                "head": key[1],
                "top phase": scheme.phase_ids[share.argmax()],
                "top share": share.max(),
                "never predicted": ", ".join(scheme.phase_ids[i] for i in np.flatnonzero(share == 0)),
                "normalised entropy": entropy(share, base=scheme.num_phases),
                "total variation": 0.5 * np.abs(share - true_share).sum(),
            }
        )
    bias_tables[name] = pd.DataFrame(rows).set_index(["model", "head"]).reindex(models, level="model")

    fig, ax = plt.subplots(figsize=(12, 3.6), layout="constrained")
    clean_axes(ax)
    plot_keys = sorted(scheme.best_keys, key=lambda key: models.index(key[0]))
    width = 0.8 / (len(plot_keys) + 1)
    positions = np.arange(scheme.num_phases)
    ax.bar(positions - 0.4 + width / 2, true_share * 100, width, color=NEUTRAL, label="ground truth")
    for index, key in enumerate(plot_keys, start=1):
        ax.bar(positions - 0.4 + width * (index + 0.5), predicted_share[key] * 100, width, color=MODEL_COLORS[key[0]], label=key[0])
    ax.set_xticks(positions, labels=scheme.phase_ids)
    ax.set_ylabel("Frames (%)")
    figure_legend(fig, [ax])
    fig.suptitle(f"Predicted against true phase distribution, {name}, best head per model, {ENSEMBLE}", x=0.01, ha="left", color=INK)
    plt.show()

bias = with_scheme(bias_tables)
bias.round(3)

## Online temporal decoding

Effect of causal smoothing (moving average, moving median, exponential moving average) and the HMM filter on frame heads and the TCN head, per window.

In [ ]:
decoding_metrics = ["f1_macro", "edit_score"]
decoding_tables, head_gains = {}, []
for name, scheme in schemes.items():
    scores = scheme.best_main.pivot_table(index="decoding", columns="model", values=decoding_metrics, aggfunc="mean")
    scores = scores.reindex(index=scheme.decoding_order)
    gain = scores - scores.loc["none"]
    scheme.best_decoding = scores["f1_macro"].idxmax()

    per_head = scheme.main.pivot_table(index="decoding", columns=["model", "head"], values="f1_macro", aggfunc="mean")
    per_head = (per_head - per_head.loc["none"]).T.groupby("head").mean().T
    head_gains.append(per_head.reindex(index=scheme.decoding_order).assign(scheme=name).set_index("scheme", append=True))

    fig, axes = plt.subplots(
        1, len(decoding_metrics), figsize=(12, 0.32 * len(scheme.decoding_order) + 1.6), layout="constrained", sharey=True, squeeze=False
    )
    for ax, metric in zip(axes[0], decoding_metrics, strict=True):
        clean_axes(ax, grid_axis="x")
        positions = np.arange(len(scheme.decoding_order))
        for model in models:
            if (metric, model) not in gain.columns:
                continue
            ax.scatter(gain[(metric, model)], positions, s=36, color=MODEL_COLORS[model], edgecolor=SURFACE, linewidth=1, zorder=3, label=model)
        ax.axvline(0, color=MUTED_INK, linewidth=1)
        ax.set_yticks(positions, labels=scheme.decoding_order)
        ax.set_ylim(len(scheme.decoding_order) - 0.5, -0.5)
        ax.spines["left"].set_visible(False)
        ax.tick_params(axis="y", length=0, labelcolor=INK)
        ax.set_xlabel(f"Change in {metric} against no decoding")
        ax.set_title(metric, loc="left", fontsize=10)
    figure_legend(fig, axes)
    fig.suptitle(f"Online decoding of head predictions, {name}, best head per model, {ENSEMBLE}", x=0.01, ha="left", color=INK)
    plt.show()

    decoding_tables[name] = pd.DataFrame(
        {
            "head": scheme.best_head,
            "best decoding": scheme.best_decoding,
            "f1_macro none": scores["f1_macro"].loc["none"],
            "f1_macro decoded": scores["f1_macro"].max(),
            "edit_score none": scores["edit_score"].loc["none"],
            "edit_score decoded": [scores[("edit_score", model)][label] for model, label in scheme.best_decoding.items()],
            "f1_macro held-out": held_out_selection(scheme.main, ["head", "decoding"]).groupby("model")["f1_macro"].mean(),
        }
    ).reindex(models)

print("mean change in macro F1 against no decoding per head, averaged over models and schemes")
display(pd.concat(head_gains).groupby(level=0, sort=False).mean()[heads].round(3))
print(
    "best decoding and decoded scores: highest mean over all surgeries (optimistic); "
    "f1_macro held-out: head and decoding chosen per test surgery on the other surgeries"
)
with_scheme(decoding_tables).round(3)

## Temporal consistency

Transitions per surgery, backward transitions, edit score and F1@k against the ground truth. Whether predictions follow the phase order.

In [ ]:
consistency_rows = []
for name, scheme in schemes.items():
    hmms = leave_one_out_hmms(scheme.labels, scheme.num_phases, scheme.run["hmm_pseudocount"])
    for surgery_id, y in scheme.labels.items():
        count, backward = transitions(y)
        consistency_rows.append(
            {"scheme": name, "model": "ground truth", "view": "ground truth", "surgery_id": surgery_id,
             "transitions": count, "backward transitions": backward, "edit_score": 1.0, "f1_at_25": 1.0}
        )
    for model, head in scheme.best_head.items():
        key = (model, head, ENSEMBLE)
        for view, label in (("no decoding", "none"), ("best decoding", scheme.best_decoding[model])):
            decoding = scheme.decodings_by_label[label]
            rows = scheme.main[(scheme.main["model"] == model) & (scheme.main["head"] == head) & (scheme.main["decoding"] == label)]
            for row in rows.itertuples():
                hmm = hmms[row.surgery_id] if decoding.method in HMM_METHODS else None
                predicted = decoding.apply(scheme.probabilities[key][row.surgery_id], FPS, hmm)
                count, backward = transitions(np.where(scheme.labels[row.surgery_id] == UNLABELLED, UNLABELLED, predicted))
                consistency_rows.append(
                    {"scheme": name, "model": model, "view": view, "surgery_id": row.surgery_id,
                     "transitions": count, "backward transitions": backward, "edit_score": row.edit_score, "f1_at_25": row.f1_at_25}
                )
consistency = pd.DataFrame(consistency_rows)

views = ["no decoding", "best decoding"]
fig, axes = plt.subplots(
    len(schemes), 2, figsize=(12, (0.42 * len(models) + 1.3) * len(schemes) + 0.8), layout="constrained", sharex="col", squeeze=False
)
for row, name in enumerate(scheme_names):
    table = consistency[consistency["scheme"] == name].groupby(["model", "view"])[CONSISTENCY_METRICS].mean()
    truth = table.loc[("ground truth", "ground truth")]
    positions = np.arange(len(models))
    for ax, metric in zip(axes[row], ("transitions", "backward transitions"), strict=True):
        clean_axes(ax, grid_axis="x")
        for offset, view in zip((-0.18, 0.18), views, strict=True):
            values = [table[metric].get((model, view), np.nan) for model in models]
            ax.barh(
                positions + offset, values, 0.34, color=[MODEL_COLORS[m] for m in models], alpha=1.0 if view == "best decoding" else 0.45,
            )
        ax.axvline(truth[metric], color=INK, linestyle="--", linewidth=1, label="ground truth")
        ax.set_xscale("symlog", linthresh=1)
        ax.set_yticks(positions, labels=models if metric == "transitions" else [])
        ax.set_ylim(len(models) - 0.5, -0.5)
        ax.tick_params(axis="y", length=0, labelcolor=INK)
        ax.set_title(f"{metric} per surgery, {name}", loc="left", fontsize=10)
for ax in axes[-1]:
    ax.set_xlabel("pale = no decoding, solid = best decoding (symlog)")
figure_legend(fig, axes)
fig.suptitle(f"Temporal consistency against the ground truth, best head per model, {ENSEMBLE}", x=0.01, ha="left", color=INK)
plt.show()

print("mean per surgery; backward transitions step to an earlier phase in the phase order")
(
    consistency.groupby(["scheme", "model", "view"])[CONSISTENCY_METRICS].mean()
    .reindex(scheme_names, level="scheme")
    .reindex(["ground truth", *models], level="model")
    .round(2)
)

## Predicted timelines

Predicted phase sequence against the ground truth per surgery for the best head per model, raw and decoded.

In [ ]:
for name, scheme in schemes.items():
    hmms = leave_one_out_hmms(scheme.labels, scheme.num_phases, scheme.run["hmm_pseudocount"])
    rows = {}
    for surgery_id, y in scheme.labels.items():
        rows[f"{surgery_id} ground truth"] = y
        for model in models:
            key = (model, scheme.best_head.get(model), ENSEMBLE)
            if key not in scheme.probabilities or surgery_id not in scheme.probabilities[key]:
                continue
            decoding = scheme.decodings_by_label[scheme.best_decoding[model]]
            hmm = hmms[surgery_id] if decoding.method in HMM_METHODS else None
            rows[f"{surgery_id} {model}"] = decoding.apply(scheme.probabilities[key][surgery_id], FPS, hmm)
    ax = plot_timeline(rows, scheme.phase_names, scheme.colors, FPS)
    ax.set_title(
        f"{name}: ground truth and each model's best head and decoding, {ENSEMBLE} "
        f"({', '.join(f'{m}: {scheme.best_head[m]}, {scheme.best_decoding[m]}' for m in models if m in scheme.best_decoding)})",
        loc="left", fontsize=9,
    )
    plt.show()

## Per-surgery variability

Spread of macro F1 across surgeries per model and head. Whether some surgeries are consistently harder.

In [ ]:
surgery_ids = sorted({s for scheme in schemes.values() for s in scheme.results["surgery_id"].unique()})
surgery_tables = {}
fig, axes = plt.subplots(
    len(schemes), 2, figsize=(1.3 * len(models) * 2 + 2.5, (0.42 * len(surgery_ids) + 1.2) * len(schemes) + 0.6),
    layout="constrained", sharey=True, squeeze=False,
)
for row, (name, scheme) in enumerate(schemes.items()):
    decoded = scheme.best_main[
        [scheme.best_decoding.get(m) == d for m, d in zip(scheme.best_main["model"], scheme.best_main["decoding"], strict=True)]
    ]
    for ax, (view, frame) in zip(axes[row], {"no decoding": scheme.best_raw, "best decoding": decoded}.items(), strict=True):
        matrix = frame.pivot_table(index="surgery_id", columns="model", values="f1_macro").reindex(index=surgery_ids, columns=models)
        heatmap(ax, matrix.to_numpy(), models, surgery_ids, f"{name}, {view}")
    minutes = pd.Series({s: len(y) / FPS / 60 for s, y in scheme.labels.items()}, name="minutes")
    table = scheme.best_raw.groupby("surgery_id")["f1_macro"].agg(["mean", "std", "min", "max"]).add_prefix("f1_macro ")
    table["f1_macro decoded mean"] = decoded.groupby("surgery_id")["f1_macro"].mean()
    table["rank"] = table["f1_macro mean"].rank(ascending=False).astype(int)
    surgery_tables[name] = pd.concat([minutes, table], axis=1).sort_values("rank")
fig.suptitle(f"Macro F1 per surgery, best head per model, {ENSEMBLE}", x=0.01, ha="left", color=INK)
plt.show()

print("per-surgery summary over models")
with_scheme(surgery_tables).round(3)

## Model comparison

Summary table of the metrics above, one row per model, head and decoding, with the configuration chosen on the other surgeries.

In [ ]:
comparison_tables = {}
for name, scheme in schemes.items():
    held_out = held_out_selection(scheme.main, ["head", "decoding"])
    table = held_out.groupby("model")[SUMMARY_METRICS].mean().add_suffix(" held-out")
    chosen = held_out.groupby("model")["config"].agg(lambda configs: configs.value_counts().index[0])
    table.insert(0, "most chosen head | decoding", chosen)
    best = scheme.main.groupby(["model", "head", "decoding"])["f1_macro"].mean()
    best = best.loc[best.groupby("model").idxmax()].reset_index(level=["head", "decoding"])
    table["best head | decoding"] = best["head"] + " | " + best["decoding"]
    table["f1_macro best"] = best["f1_macro"]
    comparison_tables[name] = table.reindex(models)
print(
    "held-out: head and decoding chosen per test surgery on the other surgeries, mean over surgeries; "
    "best: highest mean over all surgeries (optimistic)"
)
model_comparison = with_scheme(comparison_tables)
model_comparison.round(3)

## Merge scheme comparison

Macro F1, precision, IoU and accuracy per model and scheme for the best head, against the majority-class baseline of each scheme.

In [ ]:
comparison_rows = []
for name, scheme in schemes.items():
    for view, selected in (
        ("no decoding", held_out_selection(scheme.raw, ["head"])),
        ("decoded", held_out_selection(scheme.main, ["head", "decoding"])),
    ):
        summary = selected.groupby("model")[COMPARE_METRICS].mean().reset_index()
        comparison_rows += summary.assign(scheme=name, view=view).to_dict("records")
majority = baselines[baselines["baseline"] == "majority"].groupby("scheme")[COMPARE_METRICS].mean().reset_index()
comparison_rows += majority.assign(view="majority baseline", model="majority").to_dict("records")
scheme_comparison = pd.DataFrame(comparison_rows)

positions = np.arange(len(scheme_names))
fig, axes = plt.subplots(1, len(COMPARE_METRICS), figsize=(3.6 * len(COMPARE_METRICS), 3.8), layout="constrained", sharey=True)
for ax, metric in zip(axes, COMPARE_METRICS, strict=True):
    clean_axes(ax)
    for model in models:
        for view, style in (("decoded", "-"), ("no decoding", ":")):
            values = scheme_comparison[(scheme_comparison["model"] == model) & (scheme_comparison["view"] == view)].set_index("scheme")[metric]
            ax.plot(
                positions, values.reindex(scheme_names), linestyle=style, marker="o", markersize=4,
                color=MODEL_COLORS[model], label=model if view == "decoded" else None,
            )
    baseline_values = scheme_comparison[scheme_comparison["view"] == "majority baseline"].set_index("scheme")[metric]
    ax.scatter(positions, baseline_values.reindex(scheme_names), marker="_", s=300, color=MUTED_INK, zorder=3, label="majority baseline")
    ax.set_xticks(positions, labels=scheme_names, rotation=20, ha="right")
    ax.set_ylim(0, 1)
    ax.set_title(metric, loc="left", fontsize=10)
axes[0].plot([], [], color=MUTED_INK, linestyle="-", label="decoded (held-out)")
axes[0].plot([], [], color=MUTED_INK, linestyle=":", label="no decoding (held-out)")
figure_legend(fig, axes)
fig.suptitle(f"Trained heads across phase merge schemes, {ENSEMBLE}", x=0.01, ha="left", color=INK)
plt.show()

print("head and decoding chosen per test surgery on the other surgeries, mean over surgeries")
(
    scheme_comparison.pivot_table(index="model", columns=["scheme", "view"], values=["f1_macro", "iou"])
    .reindex(columns=scheme_names, level="scheme")
    .reindex([*models, "majority"])
    .round(3)
)

## Per-phase performance across merge schemes

F1 per phase and model for each scheme. Shows which merges remove the confusions.

In [ ]:
fig, axes = plt.subplots(
    1, len(schemes), figsize=(1.1 * len(models) * len(schemes) + 3, 0.45 * max(s.num_phases for s in schemes.values()) + 2),
    layout="constrained", squeeze=False,
)
for ax, (name, scheme) in zip(axes[0], schemes.items(), strict=True):
    matrix = (
        per_phase[per_phase["scheme"] == name]
        .pivot(index="phase", columns="model", values="f1")
        .reindex(index=scheme.phase_ids, columns=models)
    )
    heatmap(ax, matrix.to_numpy(), models, scheme.phase_ids, name)
fig.suptitle(f"Per-phase F1, best head per model and scheme, {ENSEMBLE}, no decoding", x=0.01, ha="left", color=INK)
plt.show()

for name, scheme in schemes.items():
    print(name)
    for phase_name in scheme.phase_names:
        print(f"  {phase_name}")

## Heads against zero-shot

Best head against the best zero-shot configuration per model and scheme: what training on labelled surgeries adds over text prompts.

In [ ]:
versus_rows = []
for name, scheme in schemes.items():
    heads_held_out = held_out_selection(scheme.main, ["head", "decoding"]).groupby("model")[COMPARE_METRICS].mean()
    versus_rows += heads_held_out.reset_index().assign(scheme=name, source="trained head").to_dict("records")
    if name in zeroshot:
        zeroshot_held_out = held_out_selection(zeroshot[name], ["variant", "decoding"]).groupby("model")[COMPARE_METRICS].mean()
        versus_rows += zeroshot_held_out.reset_index().assign(scheme=name, source="zero-shot").to_dict("records")
versus = pd.DataFrame(versus_rows)

fig, axes = plt.subplots(
    1, len(schemes), figsize=(3.4 * len(schemes), 0.5 * len(models) + 1.8), layout="constrained", sharey=True, squeeze=False
)
for ax, name in zip(axes[0], scheme_names, strict=True):
    clean_axes(ax, grid_axis="x")
    table = versus[versus["scheme"] == name].pivot(index="model", columns="source", values="f1_macro").reindex(models)
    positions = np.arange(len(models))
    if "zero-shot" in table:
        ax.hlines(positions, table["zero-shot"], table["trained head"], color=GRIDLINE, linewidth=2, zorder=1)
        ax.scatter(table["zero-shot"], positions, s=40, facecolor=SURFACE, edgecolor=MUTED_INK, zorder=2, label="zero-shot")
    ax.scatter(table["trained head"], positions, s=40, color=[MODEL_COLORS[m] for m in models], zorder=3)
    ax.scatter([], [], s=40, color=INK, label="trained head")
    ax.axvline(baseline_summary.loc[(name, "majority"), "f1_macro"], color=MUTED_INK, linestyle="--", linewidth=1, label="majority baseline")
    ax.set_yticks(positions, labels=models)
    ax.set_ylim(len(models) - 0.5, -0.5)
    ax.set_xlim(0, 1)
    ax.tick_params(axis="y", length=0, labelcolor=INK)
    ax.set_xlabel("macro F1 (held-out)")
    ax.set_title(name, loc="left", fontsize=10)
figure_legend(fig, axes)
fig.suptitle(f"Trained heads against zero-shot @ {FPS:g} fps, configuration chosen on the other surgeries", x=0.01, ha="left", color=INK)
plt.show()

versus_table = versus.pivot_table(index=["scheme", "model"], columns="source", values=["f1_macro", "accuracy"])
if ("f1_macro", "zero-shot") in versus_table.columns:
    versus_table[("f1_macro", "gain")] = versus_table[("f1_macro", "trained head")] - versus_table[("f1_macro", "zero-shot")]
print("zero-shot models without a text encoder have no zero-shot row")
versus_table.reindex(scheme_names, level="scheme").reindex(models, level="model").round(3)

## Best configurations

Ranking of model, head, ensemble, decoding and scheme by held-out macro F1, and the configuration carried into the error analysis of notebook 5.

In [ ]:
configurations = (
    pd.concat(scheme.results.assign(scheme=name) for name, scheme in schemes.items())
    .groupby(["scheme", "model", "head", "seed", "decoding"])[SUMMARY_METRICS]
    .mean()
    .sort_values("f1_macro", ascending=False)
)
print(f"top {TOP_CONFIGURATIONS} configurations by mean macro F1 over surgeries (optimistic: chosen on the test surgeries)")
display(configurations.head(TOP_CONFIGURATIONS).round(3))

held_out_rows = []
for name, scheme in schemes.items():
    held_out = held_out_selection(scheme.main, ["model", "head", "decoding"], by=())
    held_out_rows.append(
        {
            "scheme": name,
            "classes": scheme.num_phases,
            "most chosen model | head | decoding": held_out["config"].value_counts().index[0],
            **held_out[SUMMARY_METRICS].mean().add_suffix(" held-out").to_dict(),
            "majority f1_macro": baseline_summary.loc[(name, "majority"), "f1_macro"],
        }
    )
best_per_scheme = pd.DataFrame(held_out_rows).set_index("scheme")
print(
    f"per scheme: model, head and decoding chosen per test surgery on the other surgeries ({ENSEMBLE}), mean over surgeries; "
    "this is the configuration carried into the error analysis"
)
best_per_scheme.round(3)